# 03 - Hypothesis test - do electric cars lose value faster?

| Part | Content |
|---|---|
| 1 | The hypothesis and how we measure "losing value" |
| 2 | The sample: only where the groups overlap |
| 3 | A first number, without controls |
| 4 | The model: same age, same mileage, same brand |
| 5 | The statistical test |
| 6 | Robustness: does the result survive other choices? |
| 7 | What it means in euros |
| 8 | Extra metric: value retention by brand and fuel |
| 9 | Conclusion and limits |

## Setup

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.cleaning import load_raw, clean
from src.metrics import (overlap_sample, fit_depreciation_model, slopes_table, compare_slopes,
                         matched_models, simple_annual_depreciation, value_retention, OVERLAP_MAX_AGE)

pd.set_option("display.max_columns", None)
FUEL_COLORS = {"Petrol": "#5B8DEF", "Diesel": "#9AA5B1", "Hybrid": "#FFB547", "Electric": "#00B386", "Other": "#B48CFF"}
df = clean(load_raw())
print(df.shape)

(244255, 17)


---
# 1. The hypothesis and how we measure it

- **H0 (null):** at the same age and mileage, electric cars lose value at the **same rate** as combustion cars.
- **H1:** electric cars lose value **faster**.

We do not know the original price of each car, so we cannot compute "price now / price new" for one car. Instead we use the **cross-section**:
cars of different ages are on sale *today*, and the way their price falls with age tells us how fast a car loses value.

**How:** a regression of the **logarithm of the price** on age and mileage.

`log(price) = brand effect + fuel effect + slope_age[fuel] * age + slope_km[fuel] * km`

- With the log, a slope of -0.09 per year means the price is multiplied by exp(-0.09) = 0.914 every year, a loss of **8.6% per year**, whatever the price level.
  This answers "faster" in **relative** terms (notebook 02, insight 4).
- Each fuel group has **its own** age slope and km slope. H0 means: slope_age[Electric] = slope_age[Petrol].
- "Same brand": the **brand effect** lets a BMW start at a different price than a Dacia, so the slopes are not driven by the brand mix.
- "Same mileage": the km term. Age and mileage are strongly correlated (notebook 02), so both must be in.

---
# 2. The sample: only where the groups overlap

95% of the electric cars are 6 years old or less. A petrol car of 15 years has no electric equivalent, so comparing slopes over the full range
would be an extrapolation. We keep **age 0-6 years** and four fuel groups: Petrol, Diesel, Hybrid, Electric.

In [2]:
sample = overlap_sample(df)
print(f"Sample: {len(sample):,} cars out of {len(df):,}")
pd.crosstab(pd.cut(sample["age_years"], [0, 1, 2, 3, 4, 5, 6]), sample["fuel_group"])

Sample: 126,249 cars out of 244,255


fuel_group,Diesel,Electric,Hybrid,Petrol
age_years,,,,
"(0, 1]",3699,1603,3724,12712
"(1, 2]",3221,1133,2211,11687
"(2, 3]",3964,1353,2287,8419
"(3, 4]",6745,665,1690,10592
"(4, 5]",10493,525,969,15897
"(5, 6]",8512,278,563,13306


Electric cars are present in every age bucket, with at least 278 cars, and many more under 3 years. The overlap is real but **thinner at 4-6 years**
for the electric group: keep this in mind when reading the confidence intervals.

---
# 3. A first number, without controls

In [3]:
raw_loss = simple_annual_depreciation(sample).set_index("fuel_group").round(1)
raw_loss

,annual_loss_pct,listings
fuel_group,,
Diesel,14.3,36634
Electric,19.4,5558
Hybrid,5.7,11444
Petrol,9.0,72613


Fitting only `log(price) ~ age` on this sample: electric cars lose about **19% per year**, petrol about **9%** and diesel about **14%** (diesel looks bad here because diesel cars are mostly large, expensive, older models: the brand and mileage are not controlled yet). But this number mixes the effect of fuel with
the effect of brand and mileage. Let us control for them.

---
# 4. The model: same age, same mileage, same brand

In [4]:
model = fit_depreciation_model(sample, absorb="brand")
print(f"R-squared: {model.rsquared:.2f}   |   observations: {int(model.nobs):,}")

age_loss = slopes_table(model, "age_years")
age_loss[["loss_pct", "loss_pct_low", "loss_pct_high"]].round(1)

R-squared: 0.66   |   observations: 126,249


,loss_pct,loss_pct_low,loss_pct_high
fuel_group,,,
Diesel,10.3,10.0,10.6
Electric,18.7,18.0,19.4
Hybrid,8.1,7.6,8.6
Petrol,8.3,8.1,8.5


**Reading the table:** `loss_pct` is the price lost **each year of age, at the same mileage and brand**, with its 95% confidence interval.
(R-squared 0.66: age, mileage, fuel and brand explain two thirds of the variation of the log price. Power, equipment and condition explain part of the rest.)

In [5]:
plot = age_loss.reset_index().sort_values("loss_pct")
fig = go.Figure(go.Bar(
    x=plot["fuel_group"], y=plot["loss_pct"],
    error_y=dict(type="data", symmetric=False,
                 array=plot["loss_pct_high"] - plot["loss_pct"], arrayminus=plot["loss_pct"] - plot["loss_pct_low"]),
    marker_color=[FUEL_COLORS[f] for f in plot["fuel_group"]],
    text=plot["loss_pct"].round(1).astype(str) + "%", textposition="outside"))
fig.update_layout(title="Annual price loss by fuel, same age-mileage-brand (95% CI)", yaxis_title="% of price lost per year")
fig.show()

The electric bar is about **twice as high** as petrol, diesel and hybrid, and the confidence intervals do not overlap.

And the mileage effect (loss per extra 10,000 km, at the same age)?

In [6]:
slopes_table(model, "km_10k")[["loss_pct", "loss_pct_low", "loss_pct_high"]].round(2)

,loss_pct,loss_pct_low,loss_pct_high
fuel_group,,,
Diesel,2.67,2.58,2.77
Electric,-0.51,-1.17,0.15
Hybrid,1.22,0.92,1.51
Petrol,1.48,1.33,1.62


Petrol loses about 1.5% for each extra 10,000 km and diesel about 2.7%, while for **electric cars mileage has almost no effect** (the interval includes 0).
So for electric cars the price is driven mostly by **age** (technology moves fast, new models arrive with more range), not by wear.
This also means the age slope is the right place to look for their faster depreciation.

---
# 5. The statistical test

Test H0: slope_age[Electric] = slope_age[other fuel]. A negative difference means electric loses value faster.

In [7]:
tests = pd.DataFrame([compare_slopes(model, "Electric", other) for other in ["Petrol", "Diesel", "Hybrid"]])
tests.round(4)

,comparison,slope_diff,ci_low,ci_high,p_value
0,Electric vs Petrol,-0.1204,-0.1291,-0.1117,0.0
1,Electric vs Diesel,-0.0987,-0.1076,-0.0898,0.0
2,Electric vs Hybrid,-0.1229,-0.1331,-0.1127,0.0


- The difference of slopes is about **-0.12** against petrol (-0.10 against diesel and -0.12 against hybrid), with narrow 95% intervals that are far from 0.
- The p-value is practically 0 (the table shows 0.0 because of rounding; the real values are below 1e-100). We **reject H0**: at the same age, mileage and brand,
  electric cars lose value significantly faster.

**Careful with p-values:** with 126,000 cars almost any difference is "significant". What matters is the **size**: 18.7% against 8.3% per year is a big effect.

---
# 6. Robustness: does the result survive other choices?

A result that exists only for one specification is fragile. We repeat the test changing one thing at a time:

| Spec | What changes |
|---|---|
| A | main model (brand effect, age 0-6) |
| B | only cars up to **4 years**, where electric cars are most numerous |
| C | only **car models that exist both as electric and as petrol/diesel** (e.g. Fiat 500, Kia Niro, Hyundai Kona), with a **model effect** instead of brand: the closest to "same car" |
| D | without the **top 1% of prices** (supercars and rare cars) |

In [8]:
def run_spec(name, data, absorb="brand"):
    res = fit_depreciation_model(data, absorb)
    loss = slopes_table(res)["loss_pct"]
    out = {"spec": name, "cars": int(res.nobs), "EV loss %/yr": loss["Electric"],
           "Petrol loss %/yr": loss["Petrol"], "Diesel loss %/yr": loss["Diesel"]}
    for other in ["Petrol", "Diesel"]:
        t = compare_slopes(res, "Electric", other)
        out[f"EV-{other} slope diff"] = t["slope_diff"]
        out[f"p-value vs {other}"] = t["p_value"]
    return out

models_both = matched_models(df)
price_cut = sample["price_in_euro"].quantile(0.99)

specs = [
    run_spec("A main (age 0-6, brand)", sample),
    run_spec("B age 0-4", overlap_sample(df, max_age=4)),
    run_spec(f"C same model ({len(models_both)} models)", sample[sample["model"].isin(models_both)], absorb="model"),
    run_spec("D no top 1% prices", sample[sample["price_in_euro"] <= price_cut]),
]
robust = pd.DataFrame(specs).set_index("spec")
robust.round(3)

,cars,EV loss %/yr,Petrol loss %/yr,Diesel loss %/yr,EV-Petrol slope diff,p-value vs Petrol,EV-Diesel slope diff,p-value vs Diesel
spec,,,,,,,,
"A main (age 0-6, brand)",126249,18.733,8.336,10.303,-0.120,0.0,-0.099,0.0
B age 0-4,75706,19.685,9.665,12.363,-0.118,0.0,-0.087,0.0
C same model (19 models),11356,13.985,8.102,8.889,-0.066,0.0,-0.058,0.0
D no top 1% prices,124988,18.863,8.406,10.261,-0.121,0.0,-0.101,0.0


In **all four** specifications electric cars lose more per year than petrol and diesel, with p-values far below 0.05.
The size changes: in spec C (the same model sold as electric and as combustion) the gap is smaller (about 14% vs 8%) but still large.
We read it this way: part of the faster loss in the main model comes from *which* electric models exist (they are not identical to the petrol ones),
but **a robust core remains even for the same model**.

---
# 7. What it means in euros

A 40,000 EUR car, after 4 years, using the yearly losses of the main model:

In [9]:
example = age_loss[["loss_pct"]].copy()
example["value_after_4y"] = (40_000 * (1 - example["loss_pct"] / 100) ** 4).round(-2)
example["share_of_price_kept_%"] = ((1 - example["loss_pct"] / 100) ** 4 * 100).round(0)
example.drop(columns="loss_pct").loc[["Petrol", "Diesel", "Hybrid", "Electric"]]

,value_after_4y,share_of_price_kept_%
fuel_group,,
Petrol,28200.0,71.0
Diesel,25900.0,65.0
Hybrid,28500.0,71.0
Electric,17400.0,44.0


After 4 years an electric car keeps about **44%** of its price, a petrol car about **71%**. (A simplified calculation: it assumes a constant yearly %
and ignores mileage.)

---
# 8. Extra metric: value retention by brand and fuel

**Retention** = median price of cars aged 3-5 years / median price of nearly-new cars (up to 1 year), same brand and fuel.
0.60 means a 4-year-old car sells for 60% of a nearly-new one. Groups need at least 30 cars in each window.

In [10]:
ret_fuel = value_retention(df, by=["fuel_group"]).set_index("fuel_group")
ret_fuel[["price_new", "price_used", "retention"]].round(2)

,price_new,price_used,retention
fuel_group,,,
Diesel,50250.0,27949.0,0.56
Electric,45825.0,22990.0,0.50
Hybrid,34990.0,29980.0,0.86
Other,21890.0,18950.0,0.87
Petrol,29990.0,21888.0,0.73


In [11]:
ret = value_retention(df)
both = ret[ret["fuel_group"].isin(["Electric", "Petrol"])].pivot(index="brand", columns="fuel_group", values="retention").dropna()
both["gap"] = both["Electric"] - both["Petrol"]
both = both.sort_values("gap")
print(f"Brands with both an electric and a petrol retention: {len(both)}")
print(f"Brands where electric retains LESS value than petrol: {(both['gap'] < 0).sum()} of {len(both)}")
both.round(2)

Brands with both an electric and a petrol retention: 9
Brands where electric retains LESS value than petrol: 7 of 9


fuel_group,Electric,Petrol,gap
brand,,,
volkswagen,0.34,0.71,-0.37
renault,0.38,0.62,-0.24
hyundai,0.53,0.70,-0.17
nissan,0.60,0.76,-0.16
bmw,0.39,0.55,-0.15
kia,0.62,0.71,-0.09
audi,0.57,0.63,-0.06
opel,0.72,0.65,0.07
mercedes-benz,0.67,0.59,0.08


In [12]:
plot = both.reset_index().melt(id_vars="brand", value_vars=["Petrol", "Electric"], var_name="fuel", value_name="retention")
fig = px.bar(plot, x="retention", y="brand", color="fuel", barmode="group", orientation="h",
             color_discrete_map=FUEL_COLORS, title="Value retention after 3-5 years: electric vs petrol, same brand",
             labels={"retention": "price at 3-5 years / price up to 1 year"})
fig.update_layout(height=450, yaxis={"categoryorder": "array", "categoryarray": both.index[::-1]})
fig.show()

In [13]:
ret_all = ret.copy()
ret_all["brand"] = ret_all["brand"].str.title()
top = ret_all.sort_values("retention", ascending=False)
print("Highest retention:"); display(top.head(5)[["brand", "fuel_group", "retention", "n_new", "n_used"]].round(2))
print("Lowest retention:"); display(top.tail(5)[["brand", "fuel_group", "retention", "n_new", "n_used"]].round(2))

Highest retention:


,brand,fuel_group,retention,n_new,n_used
47,Mitsubishi,Petrol,1.08,120,204
36,Land-Rover,Hybrid,1.03,105,60
21,Ford,Hybrid,0.94,242,159
23,Honda,Hybrid,0.84,67,34
67,Toyota,Hybrid,0.81,1335,845


Lowest retention:


,brand,fuel_group,retention,n_new,n_used
37,Land-Rover,Petrol,0.42,162,292
8,Bmw,Electric,0.39,90,118
69,Volkswagen,Diesel,0.39,545,1807
59,Renault,Electric,0.38,89,252
70,Volkswagen,Electric,0.34,134,147


- By fuel: after 3-5 years electric cars keep about **50%** of the nearly-new price, diesel 56%, petrol 73% and hybrid 86%.
  Diesel is also low in this raw view, but it is a mix effect (large expensive models, see section 3); the controlled model of section 4 puts diesel at 10% per year, close to petrol.
- Inside the same brand, electric retains less than petrol for **7 of 9 brands** (the gap is -0.37 for Volkswagen and -0.24 for Renault). The exceptions are Opel and Mercedes-Benz.
- Retention is a **descriptive** metric: it does not control for model mix or mileage. A value above 1 (Mitsubishi petrol 1.08) means that the 3-5 year cars are a different
  mix of models from the nearly-new ones. So we use the regression (sections 4-6) for the hypothesis and retention only to **compare brands**.

---
# 9. Conclusion and limits

**Result: the hypothesis is supported.** At the same age, mileage and brand, electric cars lose about **19% of their price per year** against about **8% for petrol**,
10% for diesel and 8% for hybrid. The result holds in four specifications, including the same car model sold in both versions (14% vs 8%).

**Limits (what this analysis cannot say):**
1. **Asking prices, not sale prices.** Listings may be negotiated down, and it may happen more for some fuels.
2. **Cross-section, not a panel.** We compare cars of different ages today; we do not follow one car. If older electric cars were launched at higher prices
   (early models were sold with different range and price) part of the "depreciation" is a **generation effect**.
3. **The 2022-2023 price cuts.** Manufacturers cut the new price of electric cars, which lowers used prices at every age. This is **a cause of faster
   depreciation** in this market, but it may not repeat.
4. **Only 0-6 years.** Nothing is said for older electric cars (too few).
5. **Unobserved quality:** battery condition, equipment and trim level are not in the file.
6. **Selection:** cars that stay on sale for years may be different from the ones sold quickly.
7. **Hybrid** mixes plug-in and mild hybrids.

Next: the Dash dashboard (`app.py`) to explore these metrics by brand and fuel.